# 📊 Financial Analytics & Corporate Valuation Analysis
## of Leading Indian Companies

---

| Field | Details |
|---|---|
| **Student** | Kukku Choudhary |
| **Institution** | Samrat Ashok Technological Institute, Vidisha |
| **Program** | IBM SkillsBuild Data Analytics Internship |
| **In association with** | AICTE & BharatCares |
| **Dataset** | Financial-Analytics-data1.csv (459 records) |
| **Tools** | Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-Learn · IBM Bob AI |

---

### Notebook Structure
| Section | Description |
|---|---|
| **1** | Library Imports |
| **2** | Load & Clean Dataset |
| **3** | Dataset Overview & Shape |
| **4** | Summary Statistics |
| **5** | Chart 1 — Top 10 Companies by Market Cap Share |
| **6** | Chart 2 — Valuation Multiple Boxplot by Cap Category |
| **7** | Chart 3 — Correlation Heatmap |
| **8** | Prepare ML Features & Target |
| **9** | Train / Test Split (80 / 20) |
| **10** | Train Linear Regression Model |
| **11** | Model Evaluation — R² & RMSE |
| **12** | Chart 4 — Regression Scatter Plot |
| **13** | Key Findings & Conclusion |

---
## Section 1 — Library Imports
Import all required libraries. Install with:  
```bash
pip install -r requirements.txt
```

In [ ]:
# ── Standard data science stack ─────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Machine learning ─────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# ── Notebook display settings ────────────────────────────────────────
%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.05)
pd.set_option("display.float_format", "{:,.2f}".format)

print("All libraries imported successfully.")

---
## Section 2 — Load & Clean the Dataset

The raw CSV contains several formatting issues that must be resolved before analysis:
- `market_cap_crore` and `sales_qtr_crore` use comma separators (e.g. `"583,436.72"`)
- `market_cap_share` carries a `%` suffix (e.g. `"4.40%"`)
- `market_cap_to_sales` may contain stray whitespace

All four numerical columns are cast to `float64` after cleaning.

In [ ]:
# ── Load raw CSV ─────────────────────────────────────────────────────
df = pd.read_csv("Financial-Analytics-data1.csv")

# ── Clean: market_cap_crore ──────────────────────────────────────────
# Strip comma thousands separators, then cast to float
df["market_cap_crore"] = (
    df["market_cap_crore"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)

# ── Clean: sales_qtr_crore ───────────────────────────────────────────
df["sales_qtr_crore"] = (
    df["sales_qtr_crore"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)

# ── Clean: market_cap_to_sales ───────────────────────────────────────
# Already numeric in most rows; coerce any bad values to NaN
df["market_cap_to_sales"] = pd.to_numeric(
    df["market_cap_to_sales"].astype(str).str.strip(), errors="coerce"
)

# ── Clean: market_cap_share ──────────────────────────────────────────
# Strip the '%' suffix before converting to float
df["market_cap_share"] = (
    df["market_cap_share"]
    .astype(str)
    .str.replace("%", "", regex=False)
    .str.strip()
    .replace("nan", np.nan)
    .astype(float)
)

# ── Create ML-ready alias columns with _clean suffix ─────────────────
df["market_cap_clean"]  = df["market_cap_crore"]
df["sales_qtr_clean"]   = df["sales_qtr_crore"]

# ── Drop rows where ALL four numerical columns are null ──────────────
numerical_cols = ["market_cap_crore", "sales_qtr_crore",
                  "market_cap_to_sales", "market_cap_share"]
df.dropna(subset=numerical_cols, how="all", inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"✅ Dataset loaded and cleaned.")
print(f"   Shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nMissing values per numerical column:")
print(df[numerical_cols].isnull().sum().to_string())

---
## Section 3 — Dataset Overview & Shape

Display the first few rows and column data types to confirm the cleaning step worked correctly.

In [ ]:
# First 5 rows of the cleaned dataset
df.head()

In [ ]:
# Column names, non-null counts, and data types
df.info()

In [ ]:
# Distribution of records across market capitalisation categories
print("── market_cap_category value counts ──")
print(df["market_cap_category"].value_counts().to_string())

print("\n── sales_qtr_category value counts ──")
print(df["sales_qtr_category"].value_counts().to_string())

---
## Section 4 — Summary Statistics

Compute **mean, median, min, max, and standard deviation** for each of the four numerical attributes, followed by the full `describe()` table.

In [ ]:
# Per-column custom statistics table
stats_rows = []
for col in numerical_cols:
    s = df[col].dropna()
    stats_rows.append({
        "Column"  : col,
        "Mean"    : s.mean(),
        "Median"  : s.median(),
        "Min"     : s.min(),
        "Max"     : s.max(),
        "Std Dev" : s.std(),
        "Count"   : int(s.count()),
    })

stats_df = pd.DataFrame(stats_rows).set_index("Column")
stats_df

In [ ]:
# Full pandas describe() — includes quartiles (25%, 50%, 75%)
df[numerical_cols].describe().round(2)

---
## Section 5 — Chart 1: Top 10 Companies by Market Cap Share

A **horizontal bar chart** ranking the 10 companies with the highest `market_cap_share`.  
Each bar is annotated with the exact percentage value.  
Saved to `top_10_market_share.png`.

In [ ]:
# ── Sort descending, take top 10, reverse for top-to-bottom display ──
top10 = (
    df[["company_name", "market_cap_share"]]
    .dropna()
    .sort_values("market_cap_share", ascending=False)
    .head(10)
    .iloc[::-1]   # reverse so the #1 company appears at the top
)

fig, ax = plt.subplots(figsize=(10, 6))

# ── Draw bars ─────────────────────────────────────────────────────────
bars = ax.barh(
    top10["company_name"],
    top10["market_cap_share"],
    color=sns.color_palette("Blues_d", len(top10)),
    edgecolor="white",
)

# ── Annotate each bar with its percentage value ───────────────────────
for bar, val in zip(bars, top10["market_cap_share"]):
    ax.text(
        bar.get_width() + 0.04,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.2f}%",
        va="center", fontsize=9, color="#333333",
    )

ax.set_xlabel("Market Cap Share (%)", fontsize=11)
ax.set_title("Top 10 Companies by Market Cap Share", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("top_10_market_share.png", dpi=150)
plt.show()
print("Saved → top_10_market_share.png")

---
## Section 6 — Chart 2: Market Cap-to-Sales Ratio Boxplot

A **grouped boxplot** comparing `market_cap_to_sales` across the three market capitalisation tiers: Small Cap → Mid Cap → Large Cap.  
Extreme outliers (beyond ±3σ) are excluded from the plot for readability but retained in all statistics.  
Saved to `market_cap_to_sales_boxplot.png`.

In [ ]:
# ── Filter out extreme outliers (>3 std-devs from the mean) ───────────
mts = df[["market_cap_category", "market_cap_to_sales"]].dropna()
mu, sigma = mts["market_cap_to_sales"].mean(), mts["market_cap_to_sales"].std()
mts_clean = mts[np.abs(mts["market_cap_to_sales"] - mu) <= 3 * sigma]

# ── Enforce natural category ordering ────────────────────────────────
cat_order = ["Small Cap", "Mid Cap", "Large Cap"]
cat_order = [c for c in cat_order if c in mts_clean["market_cap_category"].unique()]

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    data=mts_clean,
    x="market_cap_category",
    y="market_cap_to_sales",
    order=cat_order,
    palette="Set2",
    linewidth=1.2,
    flierprops=dict(marker="o", markersize=4, alpha=0.5),
    ax=ax,
)

ax.set_xlabel("Market Cap Category", fontsize=11)
ax.set_ylabel("Market Cap to Sales Ratio", fontsize=11)
ax.set_title(
    "Market Cap-to-Sales Ratio Distribution\nby Market Cap Category",
    fontsize=13, fontweight="bold",
)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("market_cap_to_sales_boxplot.png", dpi=150)
plt.show()
print("Saved → market_cap_to_sales_boxplot.png")

---
## Section 7 — Chart 3: Correlation Heatmap

A **Pearson correlation heatmap** across all four numerical attributes.  
- Red cells → strong positive correlation  
- Blue cells → strong negative correlation  
- White/pale cells → weak or no linear relationship  

Saved to `correlation_matrix.png`.

In [ ]:
# ── Compute Pearson correlation matrix ────────────────────────────────
corr = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True,           # Display correlation values inside each cell
    fmt=".2f",
    cmap="coolwarm",      # Diverging palette: blue=negative, red=positive
    center=0,
    linewidths=0.5,
    linecolor="white",
    square=True,
    ax=ax,
    annot_kws={"size": 10},
)

ax.set_title("Correlation Matrix — Numerical Attributes", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=30)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.savefig("correlation_matrix.png", dpi=150)
plt.show()
print("Saved → correlation_matrix.png")

---
## Section 8 — Prepare ML Features & Target

| Role | Column | Description |
|---|---|---|
| **Feature (X)** | `sales_qtr_clean` | Quarterly sales revenue (₹ Crore) |
| **Target (y)** | `market_cap_clean` | Market capitalisation (₹ Crore) |

Scikit-Learn requires `X` to be a **2-D array** of shape `(n_samples, n_features)`.  
`.reshape(-1, 1)` converts the 1-D Series into a column vector `(n, 1)`.

In [ ]:
# ── Drop rows with NaN in either modelling column ─────────────────────
ml_df = df.dropna(subset=["sales_qtr_clean", "market_cap_clean"]).copy()
ml_df.reset_index(drop=True, inplace=True)

# ── Feature matrix (2-D) and target vector (1-D) ─────────────────────
X = ml_df["sales_qtr_clean"].values.reshape(-1, 1)   # shape: (n, 1)
y = ml_df["market_cap_clean"].values                  # shape: (n,)

print(f"Feature matrix X shape : {X.shape}")
print(f"Target vector  y shape : {y.shape}")
print(f"\nSample X values (₹ Cr) : {X[:5].flatten().tolist()}")
print(f"Sample y values (₹ Cr) : {y[:5].tolist()}")

---
## Section 9 — Train / Test Split (80% / 20%)

`random_state=42` guarantees fully reproducible splits across runs.

In [ ]:
# ── 80% training / 20% testing split ─────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Total records      : {len(X)}")
print(f"Training samples   : {len(X_train)}  ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing  samples   : {len(X_test)}   ({len(X_test)/len(X)*100:.1f}%)")

---
## Section 10 — Train Linear Regression Model

Fit `LinearRegression` on the training set and inspect the learned **slope** (β₁) and **intercept** (β₀).  
The slope represents: *"for each additional ₹1 Crore in quarterly sales, the market ascribes β₁ ₹ Crore in market capitalisation."*

In [ ]:
# ── Instantiate and fit the model on training data ────────────────────
model = LinearRegression()
model.fit(X_train, y_train)

# ── Learned parameters ────────────────────────────────────────────────
slope     = model.coef_[0]
intercept = model.intercept_

print("Model trained successfully.")
print(f"\n  Slope (β₁)     : {slope:>15,.4f}")
print(f"  Intercept (β₀) : {intercept:>15,.4f}")
print(f"\n  Equation  →  Market Cap = {slope:,.2f} × Quarterly Sales + {intercept:,.2f}")

---
## Section 11 — Model Evaluation: R² Score & RMSE

| Metric | Formula | Interpretation |
|---|---|---|
| **R² Score** | 1 − SSres/SStot | Fraction of variance in market cap explained by sales. Range: 0 → 1. |
| **RMSE** | √(Σ(y − ŷ)² / n) | Average prediction error in ₹ Crore. Lower = better. |

In [ ]:
# ── Generate predictions on the held-out test set ─────────────────────
y_pred = model.predict(X_test)

# ── Compute evaluation metrics ────────────────────────────────────────
r2   = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 50)
print("  MODEL EVALUATION — Test Set")
print("=" * 50)
print(f"  R² Score : {r2:.4f}   ({r2 * 100:.2f}% variance explained)")
print(f"  RMSE     : ₹ {rmse:,.2f} crore")
print("=" * 50)

# ── Side-by-side comparison: actual vs predicted (first 10 test rows) ─
comparison = pd.DataFrame({
    "Actual Market Cap (₹ Cr)"   : y_test[:10],
    "Predicted Market Cap (₹ Cr)": y_pred[:10],
    "Residual (₹ Cr)"            : y_test[:10] - y_pred[:10],
})
print("\nFirst 10 test predictions vs actuals:")
comparison

---
## Section 12 — Chart 4: Regression Scatter Plot

Three visual layers:
- 🔵 **Blue dots** — actual test-set market capitalisations
- 🟠 **Orange × markers** — model predictions at the same sales values
- 🔴 **Red line** — smooth fitted regression line with equation, R², and RMSE in the legend

Saved to `regression_plot.png`.

In [ ]:
# ── Smooth regression line over the full test-set range ───────────────
x_line = np.linspace(X_test.min(), X_test.max(), 300).reshape(-1, 1)
y_line = model.predict(x_line)

fig, ax = plt.subplots(figsize=(10, 6))

# Layer 1 — Actual test-set data points
ax.scatter(
    X_test, y_test,
    color="#3b82d4", alpha=0.65,
    edgecolors="white", linewidths=0.5,
    s=55, label="Actual (test set)", zorder=3,
)

# Layer 2 — Predicted values at the same x positions
ax.scatter(
    X_test, y_pred,
    color="#f97316", alpha=0.45,
    marker="x", s=40,
    label="Predicted", zorder=3,
)

# Layer 3 — Fitted regression line with annotation
ax.plot(
    x_line, y_line,
    color="#dc2626", linewidth=2,
    label=(
        f"Regression line\n"
        f"y = {slope:,.1f}·x + {intercept:,.0f}\n"
        f"R² = {r2:.4f}  |  RMSE = {rmse:,.0f}"
    ),
    zorder=4,
)

# ── Axis formatting ───────────────────────────────────────────────────
ax.set_xlabel("Quarterly Sales (₹ crore)", fontsize=12)
ax.set_ylabel("Market Capitalisation (₹ crore)", fontsize=12)
ax.set_title(
    "Linear Regression: Market Cap vs Quarterly Sales",
    fontsize=14, fontweight="bold", pad=14,
)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(fontsize=9, framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="both", linestyle="--", linewidth=0.5, alpha=0.4)

plt.tight_layout()
plt.savefig("regression_plot.png", dpi=150)
plt.show()
print("Saved → regression_plot.png")

---
## Section 13 — Key Findings & Conclusion

### 13.1 Market Concentration Findings

| Rank | Company | Market Cap Share (%) |
|---|---|---|
| 1 | Reliance Industries | ~4.40% |
| 2 | TCS | ~4.25% |
| 3 | HDFC Bank | ~3.64% |
| 4 | ITC | ~2.42% |
| 5 | HDFC | ~2.18% |
| 6 | Hindustan Unilever | ~2.18% |
| 7 | Maruti Suzuki | ~1.99% |
| 8 | Infosys | ~1.87% |
| 9 | ONGC | ~1.81% |
| 10 | Bajaj Finance | ~1.61% |
| **Total (Top 10)** | | **~16.89%** |

**Insight:** The Indian equity market exhibits significant Pareto-like concentration. A mere 10 out of 459 companies (~2.2%) control ~16.89% of total market capitalisation, with three companies alone holding ~12.29%.

### 13.2 Valuation Multiple Insights
- **Large Cap** companies display the widest IQR in `market_cap_to_sales`, reflecting heterogeneous business models (capital-heavy industrials at low multiples; asset-light IT/FMCG at high multiples).
- **Small & Mid Cap** segments show denser upper-tail outliers, suggesting speculative growth premiums.

### 13.3 Correlation Insights
- `market_cap_crore` ↔ `sales_qtr_crore`: **Strong positive** — validating the regression hypothesis.
- `market_cap_crore` ↔ `market_cap_share`: **Near-perfect** — by mathematical construction.
- `market_cap_to_sales` ↔ `sales_qtr_crore`: **Weak/negative** — high-revenue companies do not always command higher multiples.

### 13.4 Regression Model Summary
- **Feature:** Quarterly Sales (₹ Crore)
- **Target:** Market Capitalisation (₹ Crore)
- **Split:** 80% train / 20% test (`random_state=42`)
- **R²** and **RMSE** are printed in Section 11 above (values depend on the live dataset).

### 13.5 Business Recommendations
1. **Track revenue momentum** in top-10 anchor companies for index-level risk assessment.
2. **Use `market_cap_to_sales` as a relative valuation screen** — above-median multiples warrant growth-sustainability scrutiny.
3. **Extend the model** with profit margins, D/E ratio, and sector encoding to improve R².
4. **Apply ensemble models** (Random Forest, XGBoost) for Small Cap segments where linearity breaks down.
5. **Diversify beyond revenue-scale metrics** — combine with ROE and FCF yield to avoid concentration risk.

---

### Output Files Generated
| File | Description |
|---|---|
| `top_10_market_share.png` | Horizontal bar — Top 10 companies by market cap share |
| `market_cap_to_sales_boxplot.png` | Boxplot — valuation multiple by cap category |
| `correlation_matrix.png` | Pearson correlation heatmap |
| `regression_plot.png` | Scatter plot with fitted regression line |

---
*Notebook authored as part of the IBM SkillsBuild Data Analytics Internship, in association with AICTE & BharatCares.*  
*AI assistance provided by **IBM Bob AI Assistant**.*

In [ ]:
# ── Final project summary printout ───────────────────────────────────
print("=" * 55)
print("  PROJECT COMPLETE — Summary")
print("=" * 55)
print(f"  Dataset records analysed  : {len(df)}")
print(f"  Numerical columns cleaned : 4")
print(f"  Visualizations saved      : 4 PNG files")
print(f"  ML model                  : Linear Regression")
print(f"  Training samples          : {len(X_train)}")
print(f"  Testing  samples          : {len(X_test)}")
print(f"  R² Score                  : {r2:.4f}")
print(f"  RMSE                      : ₹ {rmse:,.2f} crore")
print("=" * 55)
print("  IBM SkillsBuild Internship | Kukku Choudhary")
print("=" * 55)